In [1]:
import pandas as pd
import numpy as np
from cmapPy.pandasGEXpress.parse import parse

In [2]:
labels = pd.read_csv("../data_2/sider_lincs_labels_cid.csv")

print("Unique drugs:", labels["pert_id"].nunique())

Unique drugs: 820


In [3]:
# Load siginfo
siginfo = pd.read_csv("F:/CDSS/prototype4/raw_data/GSE92742_Broad_LINCS_sig_info.txt/GSE92742_Broad_LINCS_sig_info.txt", sep="\t")

# Load your drug list (820 drugs)
drug_list = set(labels["pert_id"].unique())

print("Total drugs:", len(drug_list))

Total drugs: 820


C:\Users\ramru\AppData\Local\Temp\ipykernel_15216\3646444935.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  siginfo = pd.read_csv("F:/CDSS/prototype4/raw_data/GSE92742_Broad_LINCS_sig_info.txt/GSE92742_Broad_LINCS_sig_info.txt", sep="\t")


In [4]:
# Only keep your 820 drugs
siginfo_filtered = siginfo[
    (siginfo["pert_type"] == "trt_cp") &
    (siginfo["pert_id"].isin(drug_list))
].copy()

print("Rows after filtering:", len(siginfo_filtered))
print("Unique drugs:", siginfo_filtered["pert_id"].nunique())

Rows after filtering: 19851
Unique drugs: 820


In [5]:
print(siginfo_filtered["pert_id"].nunique())  # should be ~820
print(len(siginfo_filtered))                 # ~19K
print(siginfo_filtered.groupby("pert_id").size().describe())

820
19851
count     820.000000
mean       24.208537
std        52.372519
min         2.000000
25%         6.000000
50%        17.000000
75%        27.000000
max      1323.000000
dtype: float64


In [6]:
sig_ids = siginfo_filtered["sig_id"].unique().tolist()

print(len(sig_ids))  # should be ~19K

19851


In [7]:
gctx_meta = parse(
    "F:/CDSS/prototype4/raw_data/level5_beta_trt_cp_n720216x12328.gctx",
    cid=None,
    rid=[]   # no genes → only metadata
)

all_cols = list(gctx_meta.col_metadata_df.index)

print(len(all_cols))  # ~720K

C:\Users\ramru\AppData\Local\Programs\Python\Python311\Lib\site-packages\cmapPy\pandasGEXpress\parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
C:\Users\ramru\AppData\Local\Programs\Python\Python311\Lib\site-packages\cmapPy\pandasGEXpress\parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


720216


In [ ]:
sig_ids_set = set(sig_ids)
valid_ids = [cid for cid in all_cols if cid in sig_ids_set]


print(len(valid_ids))  

17922


In [9]:
geneinfo = pd.read_csv("F:/CDSS/prototype4/raw_data/geneinfo_beta.txt", sep="\t")

# keep landmark genes
landmark = geneinfo[geneinfo["feature_space"] == "landmark"]

landmark_gene_ids = landmark["gene_id"].astype(str).tolist()

print("Landmark genes:", len(landmark_gene_ids))  # should be ~978

Landmark genes: 978


In [10]:
import contextlib
import os
import sys

@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, "w") as fnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = fnull
        sys.stderr = fnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

In [11]:
gctx_path= "F:/CDSS/prototype4/raw_data/level5_beta_trt_cp_n720216x12328.gctx"
chunks = []
chunk_size = 200

for i in range(0, len(valid_ids), chunk_size):
    batch_ids = valid_ids[i:i+chunk_size]

    try:
        with suppress_output():
            gctoo = parse(
                gctx_path,
                cid=batch_ids,
                rid=landmark_gene_ids
            )

        chunks.append(gctoo.data_df)
        print(f"Loaded batch {i} | cols: {gctoo.data_df.shape[1]}")

    except:
        print(f"Batch {i} failed")

Loaded batch 0 | cols: 200
Loaded batch 200 | cols: 200
Loaded batch 400 | cols: 200
Loaded batch 600 | cols: 200
Loaded batch 800 | cols: 200
Loaded batch 1000 | cols: 200
Loaded batch 1200 | cols: 200
Loaded batch 1400 | cols: 200
Loaded batch 1600 | cols: 200
Loaded batch 1800 | cols: 200
Loaded batch 2000 | cols: 200
Loaded batch 2200 | cols: 200
Loaded batch 2400 | cols: 200
Loaded batch 2600 | cols: 200
Loaded batch 2800 | cols: 200
Loaded batch 3000 | cols: 200
Loaded batch 3200 | cols: 200
Loaded batch 3400 | cols: 200
Loaded batch 3600 | cols: 200
Loaded batch 3800 | cols: 200
Loaded batch 4000 | cols: 200
Loaded batch 4200 | cols: 200
Loaded batch 4400 | cols: 200
Loaded batch 4600 | cols: 200
Loaded batch 4800 | cols: 200
Loaded batch 5000 | cols: 200
Loaded batch 5200 | cols: 200
Loaded batch 5400 | cols: 200
Loaded batch 5600 | cols: 200
Loaded batch 5800 | cols: 200
Loaded batch 6000 | cols: 200
Loaded batch 6200 | cols: 200
Loaded batch 6400 | cols: 200
Loaded batch 6600

In [12]:
X = pd.concat(chunks, axis=1)
print(X.shape)
print("Loaded signatures:", X.shape[1])
print("Expected valid signatures:", len(valid_ids))

(978, 17922)
Loaded signatures: 17922
Expected valid signatures: 17922


In [13]:
X = X.T   # rows = sig_id, cols = genes
sig_to_pert = siginfo_filtered.set_index("sig_id")["pert_id"]

X["pert_id"] = X.index.map(sig_to_pert)

print("Missing pert_id mappings:", X["pert_id"].isna().sum())

X = X.dropna(subset=["pert_id"])

Missing pert_id mappings: 0


In [14]:
def cap_samples(df, max_samples=50):
    return df.sample(min(len(df), max_samples), random_state=42)

X = X.groupby("pert_id", group_keys=False).apply(cap_samples)

grouped = X.groupby("pert_id")

Gdrug_mu = grouped.mean()
Gdrug_sigma = grouped.std().fillna(0)

C:\Users\ramru\AppData\Local\Temp\ipykernel_15216\1567588911.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  X = X.groupby("pert_id", group_keys=False).apply(cap_samples)


In [15]:
print(Gdrug_mu.shape)

(820, 978)


In [ ]:
# pd.DataFrame({
#     "gene_idx": range(len(Gdrug_mu.columns)),
#     "GeneSymbol": Gdrug_mu.columns
# }).to_csv("../data_2/gene_index_lincs.csv", index=False)

In [29]:
# Building correct gene index for Gdrug columns.
# Gdrug_mu.columns currently contain the LINCS gene_id values used in rid=landmark_gene_ids,
# not human-readable gene symbols.

gene_ids_in_gdrug = pd.Series(Gdrug_mu.columns.astype(str), name="GeneID")

# Keep only the columns we need from geneinfo
gene_map = geneinfo.copy()

gene_map["GeneID"] = gene_map["gene_id"].astype(str)

# Try to locate the symbol column in geneinfo_beta.txt
possible_symbol_cols = ["gene_symbol", "pr_gene_symbol", "gene_title", "gene_short_name"]
symbol_col = None

for col in possible_symbol_cols:
    if col in gene_map.columns:
        symbol_col = col
        break

if symbol_col is None:
    raise ValueError(f"Could not find a gene symbol column in geneinfo. Available columns: {gene_map.columns.tolist()}")

gene_map["GeneSymbol"] = gene_map[symbol_col].astype(str).str.upper().str.strip()

gene_index_lincs = pd.DataFrame({
    "gene_idx": range(len(gene_ids_in_gdrug)),
    "GeneID": gene_ids_in_gdrug.values
}).merge(
    gene_map[["GeneID", "GeneSymbol"]].drop_duplicates(),
    on="GeneID",
    how="left"
)

print("Rows in gene_index_lincs:", len(gene_index_lincs))
print("Missing GeneSymbol mappings:", gene_index_lincs["GeneSymbol"].isna().sum())

gene_index_lincs.to_csv("../data_2/gene_index_lincs.csv", index=False)
gene_index_lincs.head()

Rows in gene_index_lincs: 978
Missing GeneSymbol mappings: 0


,gene_idx,GeneID,GeneSymbol
0,0,10007,GNPDA1
1,1,1001,CDH3
2,2,10013,HDAC6
3,3,10038,PARP2
4,4,10046,MAMLD1


In [17]:
drug_order = sorted(Gdrug_mu.index)

Gdrug_mu = Gdrug_mu.loc[drug_order]
Gdrug_sigma = Gdrug_sigma.loc[drug_order]

In [18]:
Gdrug_mu_np = Gdrug_mu.values
Gdrug_sigma_np = Gdrug_sigma.values

In [19]:
eps = 1e-6
Gdrug_eff = Gdrug_mu_np / (Gdrug_sigma_np + eps)

# stabilize
Gdrug_eff = np.clip(Gdrug_eff, -10, 10)

In [20]:
label_drugs = set(labels["pert_id"].unique())
gdrug_drugs = set(drug_order)

print("Drugs in labels:", len(label_drugs))
print("Drugs in Gdrug:", len(gdrug_drugs))
print("Missing from Gdrug:", len(label_drugs - gdrug_drugs))

Drugs in labels: 820
Drugs in Gdrug: 820
Missing from Gdrug: 0


In [21]:
np.save("Gdrug_mu.npy", Gdrug_mu_np)
np.save("Gdrug_sigma.npy", Gdrug_sigma_np)
np.save("Gdrug_eff.npy", Gdrug_eff)

pd.DataFrame({
    "drug_idx": range(len(drug_order)),
    "pert_id": drug_order
}).to_csv("drug_index.csv", index=False)

In [1]:
print("NaNs:", np.isnan(Gdrug_mu_np).sum())
print("Drugs:", Gdrug_mu.shape[0])
print("Genes:", Gdrug_mu.shape[1])

NameError: name 'np' is not defined

In [23]:
print(Gdrug_mu.index[:5])

Index(['BRD-A00827783', 'BRD-A01320529', 'BRD-A01346607', 'BRD-A01643550',
       'BRD-A02180903'],
      dtype='object', name='pert_id')


In [24]:
print(Gdrug_mu.columns[:5])

Index(['10007', '1001', '10013', '10038', '10046'], dtype='object', name='rid')
